# Speech Recognition

**Speech Recognition**, or **Automatic Speech Recognition (ASR)**, is the task of converting spoken audio into a sequence of text:

$$
\boxed{
\text{Speech Audio}
\rightarrow
\text{Text}
}
$$

For example:

> Audio: “Hello, how are you?”

$$
\downarrow
$$

> `"Hello, how are you?"`

Speech Recognition is naturally a **sequence-to-sequence** problem because both the input and output are sequences, but their lengths are usually different:

$$
\boxed{
x^{\langle1\rangle},\ldots,x^{\langle T_x\rangle}
\rightarrow
y^{\langle1\rangle},\ldots,y^{\langle T_y\rangle}
}
$$

and typically:

$$
T_x\gg T_y
$$

A few seconds of audio may contain hundreds or thousands of acoustic frames, while the transcript may contain only a few dozen characters or words.

---

# 1. From Audio to a Sequence

The raw speech signal is a continuous-time waveform:

$$
x(t)
$$

A speech recognition system usually converts the waveform into a sequence of acoustic representations such as spectrograms or other acoustic features.

The input can therefore be represented as:

$$
\boxed{
x^{\langle1\rangle},
x^{\langle2\rangle},
\ldots,
x^{\langle T_x\rangle}
}
$$

where each timestep corresponds to a short time interval of the audio.

For example:

$$
x^{\langle t\rangle}\in\mathbb R^d
$$

can represent the acoustic features extracted from a short audio frame.

The basic pipeline is:

$$
\boxed{
\text{Audio Waveform}
\rightarrow
\text{Acoustic Features}
\rightarrow
\text{Input Sequence}
}
$$

---

# 2. Why Speech Recognition Is Difficult

Unlike sentiment classification, where we have:

$$
\text{Many-to-One}
$$

Speech Recognition is a sequence-to-sequence task:

$$
\boxed{
\text{Many-to-Many}
}
$$

but with:

$$
\boxed{
T_x\neq T_y
}
$$

For example, the word:

> `hello`

may correspond to many acoustic frames:

$$
x^{\langle1\rangle},\ldots,x^{\langle100\rangle}
$$

while the transcript contains only:

$$
[\text{h},\text{e},\text{l},\text{l},\text{o}]
$$

Therefore, the model must learn:

$$
\boxed{
\text{Long Acoustic Sequence}
\rightarrow
\text{Short Text Sequence}
}
$$

without being explicitly told which acoustic frames correspond to which output characters.

This is the **alignment problem**.

---

# 3. The Alignment Problem

Suppose the audio represents:

> `cat`

and contains:

$$
x_1,x_2,\ldots,x_{100}
$$

The model would ideally learn something like:

$$
x_{1:30}
\rightarrow
\text{c}
$$

$$
x_{31:60}
\rightarrow
\text{a}
$$

$$
x_{61:100}
\rightarrow
\text{t}
$$

However, the training dataset may only provide:

$$
\boxed{
\text{Audio}
\rightarrow
\text{Transcript}
}
$$

and not:

$$
\boxed{
\text{Audio Frames}
\rightarrow
\text{Character-Level Alignment}
}
$$

Therefore, the model must learn the alignment implicitly.

This is one of the main motivations for **Connectionist Temporal Classification (CTC)**.

---

# 4. Connectionist Temporal Classification (CTC)

CTC allows a model to learn from:

$$
\boxed{
\text{Audio}
+
\text{Transcript}
}
$$

without requiring an explicit frame-level alignment.

At each input timestep, the model predicts a distribution over the output vocabulary plus a special **blank** symbol:

$$
\mathcal V'
=
\mathcal V
\cup
\{\text{blank}\}
$$

For example:

$$
\mathcal V
=
\{\text{a},\text{b},\ldots,\text{z},\text{space}\}
$$

and:

$$
\boxed{
\mathcal V'
=
\{\text{a},\text{b},\ldots,\text{z},\text{space},\text{blank}\}
}
$$

The model produces:

$$
P(c_t\mid x)
$$

at every acoustic timestep.

---

# 5. CTC Paths and the Blank Symbol

Suppose the intended transcript is:

$$
\text{cat}
$$

A possible CTC path might be:

$$
[
\text{blank},
\text{c},
\text{c},
\text{blank},
\text{a},
\text{a},
\text{blank},
\text{t},
\text{t}
]
$$

CTC maps this path to the transcript using two main rules.

First, consecutive repeated tokens are merged:

$$
[
\text{blank},
\text{c},
\text{c},
\text{blank},
\text{a},
\text{a},
\text{blank},
\text{t},
\text{t}
]
$$

becomes:

$$
[
\text{blank},
\text{c},
\text{blank},
\text{a},
\text{blank},
\text{t}
]
$$

Then the blank symbols are removed:

$$
[
\text{blank},
\text{c},
\text{blank},
\text{a},
\text{blank},
\text{t}
]
\rightarrow
[
\text{c},\text{a},\text{t}
]
$$

giving:

$$
\boxed{
\text{cat}
}
$$

The blank therefore represents that no new output token is emitted at that timestep.

---

# 6. Multiple Paths Can Produce the Same Transcript

This is a key idea in CTC.

Many different paths can collapse to the same transcript.

For example, the transcript:

$$
\text{cat}
$$

can be produced by paths such as:

$$
[\text{c},\text{a},\text{t}]
$$

or:

$$
[\text{c},\text{c},\text{a},\text{t}]
$$

or:

$$
[\text{blank},\text{c},\text{c},\text{blank},\text{a},\text{t}]
$$

or:

$$
[\text{c},\text{blank},\text{a},\text{a},\text{blank},\text{t}]
$$

All of these collapse to:

$$
\boxed{
\text{cat}
}
$$

Therefore, the probability of a transcript is the sum of the probabilities of **all valid paths** that collapse to that transcript:

$$
\boxed{
P(y\mid x)
=
\sum_{\pi\in\mathcal B^{-1}(y)}
P(\pi\mid x)
}
$$

where:

- $\pi$ is a CTC path;
- $\mathcal B$ is the CTC collapse function.

This is how CTC handles the unknown alignment.

---

# 7. CTC Objective

Suppose:

$$
x=
(x^{\langle1\rangle},\ldots,x^{\langle T_x\rangle})
$$

is the audio sequence and:

$$
y
$$

is the target transcript.

CTC wants to maximize:

$$
P(y\mid x)
$$

which is obtained by summing over all valid alignments:

$$
\boxed{
P(y\mid x)
=
\sum_{\pi\in\mathcal B^{-1}(y)}
P(\pi\mid x)
}
$$

The corresponding loss is:

$$
\boxed{
L_{\text{CTC}}
=
-\log P(y\mid x)
}
$$

The number of valid paths can be very large, so they are not explicitly enumerated.

Instead, CTC uses **dynamic programming** to compute the total probability efficiently.

The key idea is:

$$
\boxed{
\text{CTC marginalizes over unknown alignments}
}
$$

rather than requiring the training dataset to provide them.

---

# 8. Speech Recognition Architecture with CTC

A basic CTC-based system can be viewed as:

$$
\boxed{
\text{Audio}
\rightarrow
\text{Acoustic Features}
\rightarrow
\text{Sequence Model}
\rightarrow
\text{CTC Output}
\rightarrow
\text{Transcript}
}
$$

The sequence model may be:

$$
\text{RNN}
$$

or:

$$
\text{LSTM / GRU}
$$

or more modern sequence architectures.

The model produces a probability distribution at every acoustic timestep:

$$
x^{\langle1\rangle}
\rightarrow
P_1
$$

$$
x^{\langle2\rangle}
\rightarrow
P_2
$$

$$
\vdots
$$

$$
x^{\langle T_x\rangle}
\rightarrow
P_{T_x}
$$

Each $P_t$ is a distribution over:

$$
\mathcal V'
=
\mathcal V\cup\{\text{blank}\}
$$

CTC then combines those distributions to compute:

$$
P(y\mid x)
$$

---

# 9. Training vs. Inference

These two stages should be distinguished clearly.

### Training

The model receives:

$$
(\text{audio},\text{transcript})
$$

and minimizes:

$$
\boxed{
L_{\text{CTC}}
=
-\log P(y\mid x)
}
$$

The model therefore learns to assign high probability to the target transcript through its possible alignments.

### Inference

For a new audio sequence $x$, we need to find:

$$
\boxed{
y^*
=
\arg\max_yP(y\mid x)
}
$$

This is now a decoding/search problem.

Therefore:

$$
\boxed{
\text{Training}
\rightarrow
\text{learn }P(y\mid x)
}
$$

while:

$$
\boxed{
\text{Inference}
\rightarrow
\text{search for the best }y
}
$$

---

# 10. Greedy Decoding

The simplest inference strategy is to select the most likely character at each acoustic timestep:

$$
\boxed{
c_t
=
\arg\max_c
P(c\mid x_t)
}
$$

For example, the model may produce:

$$
[
\text{blank},
\text{c},
\text{c},
\text{blank},
\text{a},
\text{a},
\text{blank},
\text{t}
]
$$

After CTC collapsing:

$$
[
\text{blank},
\text{c},
\text{c},
\text{blank},
\text{a},
\text{a},
\text{blank},
\text{t}
]
$$

$$
\rightarrow
[
\text{c},\text{a},\text{t}
]
$$

giving:

$$
\boxed{
\text{cat}
}
$$

Greedy decoding is fast, but it does not necessarily find:

$$
\arg\max_yP(y\mid x)
$$

because it makes local decisions independently.

---

# 11. Connection to Beam Search

Inference therefore connects directly to the Beam Search topic.

The model produces:

$$
\boxed{
\text{Per-timestep acoustic probabilities}
}
$$

but we ultimately want:

$$
\boxed{
y^*
=
\arg\max_yP(y\mid x)
}
$$

Greedy decoding keeps one path.

Beam Search keeps multiple candidate paths:

$$
\boxed{
\text{CTC}
\rightarrow
\text{Beam Search}
\rightarrow
\text{Better sequence decoding}
}
$$

A practical decoder can also combine acoustic evidence with a language model.

Conceptually:

$$
\boxed{
\text{Acoustic Evidence}
+
\text{Language Information}
\rightarrow
\text{Final Transcript}
}
$$

This allows the decoder to prefer linguistically plausible sequences when the acoustic evidence is ambiguous.

---

# 12. Acoustic Model and Language Model

Speech recognition involves two types of information.

### Acoustic information

The acoustic model answers:

> What sounds are present in the audio?

This is represented by probabilities such as:

$$
P(c_t\mid x)
$$

### Language information

A language model answers:

> Which text sequences are plausible in the language?

At a high level:

$$
P(y)
$$

captures the probability of a text sequence.

A classical probabilistic view is:

$$
\boxed{
P(y\mid x)
\propto
P(x\mid y)P(y)
}
$$

where:

- $P(x\mid y)$ reflects acoustic compatibility;
- $P(y)$ reflects linguistic plausibility.

The precise way these components are combined depends on the ASR architecture and decoding framework, but the central intuition is:

$$
\boxed{
\text{What was heard}
+
\text{What makes sense linguistically}
}
$$

---

# 13. Why Speech Recognition Is Challenging

The mapping:

$$
\text{Audio}
\rightarrow
\text{Text}
$$

is complicated by several factors:

$$
\boxed{
\text{Different Speakers}
}
$$

Different voices, accents, pronunciation patterns, and speaking styles.

$$
\boxed{
\text{Noise}
}
$$

Background noise and recording conditions can distort acoustic signals.

$$
\boxed{
\text{Variable Speaking Rate}
}
$$

The same word can occupy very different numbers of acoustic frames.

$$
\boxed{
\text{Coarticulation}
}
$$

The acoustic realization of a sound depends on neighboring sounds.

$$
\boxed{
\text{Acoustic Ambiguity}
}
$$

Different text sequences can produce acoustically similar signals.

Therefore, successful ASR requires:

$$
\boxed{
\text{Acoustic Representation}
+
\text{Sequence Modeling}
+
\text{Alignment Handling}
+
\text{Decoding}
}
$$

---

# Final Mental Model

Speech Recognition with CTC can be summarized as:

$$
\boxed{
\text{Audio}
\rightarrow
\text{Acoustic Features}
\rightarrow
\text{Sequence Model}
\rightarrow
\text{Per-Timestep Character Probabilities}
\rightarrow
\text{CTC}
\rightarrow
\text{Transcript}
}
$$

The fundamental challenge is:

$$
\boxed{
T_x\gg T_y
}
$$

with unknown frame-to-text alignment.

CTC solves this by allowing many possible alignments:

$$
\boxed{
\text{Many Possible Paths}
\rightarrow
\text{Collapse to Transcript}
\rightarrow
\text{Sum Their Probabilities}
}
$$

with:

$$
\boxed{
P(y\mid x)
=
\sum_{\pi\in\mathcal B^{-1}(y)}
P(\pi\mid x)
}
$$

and:

$$
\boxed{
L_{\text{CTC}}
=
-\log P(y\mid x)
}
$$

At inference:

$$
\boxed{
y^*
=
\arg\max_yP(y\mid x)
}
$$

which can be solved approximately with:

$$
\boxed{
\text{Greedy Decoding}
}
$$

or:

$$
\boxed{
\text{Beam Search}
}
$$

The complete mental model is:

$$
\boxed{
\text{Speech}
\rightarrow
\text{Acoustic Sequence}
\rightarrow
\text{Unknown Alignment}
\rightarrow
\text{CTC}
\rightarrow
\text{Sequence Probability}
\rightarrow
\text{Decoding}
}
$$

# Trigger Word Detection

**Trigger Word Detection** is the task of detecting a specific word or phrase in a continuous audio stream in order to activate another system.

For example:

> “Hey Siri, what is the weather today?”

The system needs to detect:

$$
\boxed{
\text{Hey Siri}
}
$$

and then activate a larger Speech Recognition or Voice Assistant system.

The overall pipeline is:

$$
\boxed{
\text{Continuous Audio}
\rightarrow
\text{Trigger Word Detection}
\rightarrow
\text{Activation}
\rightarrow
\text{Speech Recognition}
}
$$

Unlike Speech Recognition, Trigger Word Detection does **not** need to transcribe the entire audio. It only needs to determine **whether and when the trigger word occurs**.

---

# 1. Trigger Word Detection as a Sequence Problem

Suppose the audio is represented as a sequence:

$$
x^{\langle1\rangle},
x^{\langle2\rangle},
\ldots,
x^{\langle T_x\rangle}
$$

A sequence model processes these acoustic features and produces an output at each timestep:

$$
\hat y^{\langle1\rangle},
\hat y^{\langle2\rangle},
\ldots,
\hat y^{\langle T_x\rangle}
$$

where:

$$
y^{\langle t\rangle}\in\{0,1\}
$$

can represent:

$$
0=\text{no trigger}
$$

and:

$$
1=\text{trigger detected}
$$

Thus, Trigger Word Detection can be viewed as a **temporal sequence-labeling problem**:

$$
\boxed{
\text{Audio Sequence}
\rightarrow
\text{Detection at Each Timestep}
}
$$

For example:

$$
[0,0,0,0,1,1,1,0,0,\ldots]
$$

indicates that the trigger word occurred around that region of the audio.

---

# 2. Basic RNN Architecture

A simple recurrent architecture is:

$$
x^{\langle1\rangle}
\rightarrow
a^{\langle1\rangle}
\rightarrow
\hat y^{\langle1\rangle}
$$

$$
x^{\langle2\rangle}
\rightarrow
a^{\langle2\rangle}
\rightarrow
\hat y^{\langle2\rangle}
$$

$$
\vdots
$$

$$
x^{\langle T_x\rangle}
\rightarrow
a^{\langle T_x\rangle}
\rightarrow
\hat y^{\langle T_x\rangle}
$$

The recurrent state is:

$$
\boxed{
a^{\langle t\rangle}
=
f
\left(
a^{\langle t-1\rangle},
x^{\langle t\rangle}
\right)
}
$$

and the detection probability is:

$$
\boxed{
\hat y^{\langle t\rangle}
=
\sigma
\left(
W_ya^{\langle t\rangle}+b_y
\right)
}
$$

Thus:

$$
\hat y^{\langle t\rangle}
\approx
P(\text{trigger at timestep }t)
$$

---

# 3. Constructing the Training Labels

One important difficulty is that the training dataset usually provides:

$$
\text{Audio}
+
\text{Trigger Word Presence}
$$

rather than an exact label for every acoustic frame.

Suppose the audio contains:

> “Good morning, Hey Siri, what is the weather?”

We want the model to know where the trigger occurs.

A simplified target sequence might be:

$$
[0,0,0,0,1,1,1,0,0,\ldots]
$$

where:

- before the trigger: $y^{\langle t\rangle}=0$;
- around the trigger: $y^{\langle t\rangle}=1$.

In practice, it is often useful to assign a **small positive region** rather than a single positive timestep because the precise temporal boundary of the trigger is not perfectly defined.

The important idea is:

$$
\boxed{
\text{Audio}
\rightarrow
\text{Temporal Labels}
}
$$

so the model can learn not only whether the trigger exists, but approximately **when it occurs**.

---

# 4. Training Objective

At each timestep, the model performs binary classification.

The binary cross-entropy loss is:

$$
\boxed{
L^{\langle t\rangle}
=
-
\left[
y^{\langle t\rangle}\log\hat y^{\langle t\rangle}
+
(1-y^{\langle t\rangle})
\log(1-\hat y^{\langle t\rangle})
\right]
}
$$

The sequence-level loss can be:

$$
\boxed{
L
=
\sum_{t=1}^{T_x}
L^{\langle t\rangle}
}
$$

or its average:

$$
L
=
\frac{1}{T_x}
\sum_{t=1}^{T_x}
L^{\langle t\rangle}
$$

During backpropagation through time, the model updates its recurrent parameters and output layer so that trigger-related regions receive higher probabilities.

The training process is:

$$
\boxed{
\text{Audio}
\rightarrow
\text{Sequence Model}
\rightarrow
\text{Per-Timestep Probabilities}
\rightarrow
\text{Binary Loss}
\rightarrow
\text{BPTT}
\rightarrow
\text{Parameter Update}
}
$$

---

# 5. The Major Problem: Class Imbalance

This is one of the most important practical difficulties.

In a long audio stream, almost all timesteps are negative:

$$
y^{\langle t\rangle}=0
$$

and only a very small region corresponds to the trigger:

$$
y^{\langle t\rangle}=1
$$

Conceptually:

$$
\boxed{
\text{Many Negative Timesteps}
+
\text{Very Few Positive Timesteps}
}
$$

For example:

$$
99.5\%
\text{ negative}
\qquad
0.5\%
\text{ positive}
$$

A model that always predicts:

$$
\hat y^{\langle t\rangle}=0
$$

could still achieve very high accuracy while completely failing its actual purpose.

Therefore:

$$
\boxed{
\text{Accuracy alone is not a sufficient metric}
}
$$

The model must be evaluated carefully for false alarms and missed detections.

---

# 6. False Positive vs. False Negative

### False Positive

The model detects a trigger when there is none:

$$
y=0,\qquad \hat y=1
$$

Example:

> The user never says “Hey Siri”, but the system activates anyway.

This is a:

$$
\boxed{
\text{False Alarm}
}
$$

Too many false positives make the system annoying because it activates unexpectedly.

### False Negative

The user actually says the trigger word:

$$
y=1
$$

but the model fails to detect it:

$$
\hat y=0
$$

This is a:

$$
\boxed{
\text{Missed Detection}
}
$$

Too many false negatives make the system appear unresponsive.

Therefore, deployment requires balancing:

$$
\boxed{
\text{False Positive}
\leftrightarrow
\text{False Negative}
}
$$

---

# 7. Thresholding and Post-Processing

The model produces a continuous probability:

$$
\hat y^{\langle t\rangle}\in(0,1)
$$

We can define a threshold:

$$
\boxed{
\hat y^{\langle t\rangle}>\tau
\Rightarrow
\text{trigger detected}
}
$$

For example:

$$
\tau=0.5
$$

If:

$$
\hat y^{\langle t\rangle}=0.9
$$

the timestep is considered positive.

However, simply thresholding every timestep can produce many consecutive positive predictions.

For example:

$$
[0,0,1,1,1,1,0,0]
$$

should normally correspond to **one trigger event**, not four separate activations.

Therefore practical systems apply post-processing:

$$
\boxed{
\text{Positive Region}
\rightarrow
\text{One Trigger Event}
}
$$

A cooldown period can also be applied after detection to prevent repeated activations.

---

# 8. Why Trigger Detection Uses a Small Model

A trigger detector often runs **continuously** while the device is waiting for activation.

Therefore it should have:

$$
\boxed{
\text{Low Latency}
}
$$

$$
\boxed{
\text{Low Computational Cost}
}
$$

and:

$$
\boxed{
\text{Low Power Consumption}
}
$$

The system should not continuously run a large Speech Recognition model just to check whether the user said a trigger phrase.

A more practical architecture is:

$$
\boxed{
\text{Always-On Small Detector}
\rightarrow
\text{Trigger Detected}
\rightarrow
\text{Activate Larger ASR Model}
}
$$

This separation is an important engineering idea.

---

# 9. Streaming Nature

Trigger Word Detection is usually a **streaming** problem.

The model receives audio continuously:

$$
x^{\langle1\rangle}
\rightarrow
x^{\langle2\rangle}
\rightarrow
x^{\langle3\rangle}
\rightarrow
\cdots
$$

At each new timestep, it updates its internal state:

$$
a^{\langle t\rangle}
=
f
\left(
a^{\langle t-1\rangle},
x^{\langle t\rangle}
\right)
$$

and produces:

$$
\hat y^{\langle t\rangle}
$$

The system therefore does not need to wait for the entire recording to finish.

Instead:

$$
\boxed{
\text{New Audio}
\rightarrow
\text{Update State}
\rightarrow
\text{Predict}
\rightarrow
\text{Check Trigger}
}
$$

This is why latency is so important.

---

# 10. Data Augmentation and Robustness

A real voice assistant must work under many conditions:

- background noise;
- music;
- multiple speakers;
- different microphones;
- different distances;
- different speaking speeds;
- different environments.

Therefore training data should cover realistic conditions.

A common strategy is to create additional training examples by combining clean speech with background sounds:

$$
\boxed{
\text{Clean Speech}
+
\text{Background Noise}
\rightarrow
\text{Robust Training Example}
}
$$

The goal is:

$$
\boxed{
\text{Training Distribution}
\approx
\text{Real Deployment Distribution}
}
$$

This helps the detector avoid becoming overly sensitive to the clean conditions present in the training data.

---

# 11. Trigger Word Detection vs. Speech Recognition

These tasks should be clearly distinguished.

### Speech Recognition

The goal is:

$$
\boxed{
\text{Audio}
\rightarrow
\text{Full Transcript}
}
$$

For example:

> “Hey Siri, play some music.”

becomes:

$$
\text{"Hey Siri, play some music."}
$$

### Trigger Word Detection

The goal is:

$$
\boxed{
\text{Audio}
\rightarrow
\text{Trigger / No Trigger}
}
$$

It does not need to understand or transcribe the entire utterance.

A typical voice-assistant pipeline is:

$$
\boxed{
\text{Continuous Audio}
\rightarrow
\text{Trigger Detector}
\rightarrow
\text{Activation}
\rightarrow
\text{Speech Recognition}
}
$$

Thus Trigger Word Detection is a **front-end activation mechanism**, while Speech Recognition is responsible for full transcription.

---

# Final Mental Model

Trigger Word Detection can be summarized as:

$$
\boxed{
\text{Continuous Audio}
\rightarrow
\text{Sequence Model}
\rightarrow
P(\text{Trigger at each timestep})
\rightarrow
\text{Threshold / Post-Processing}
\rightarrow
\text{Trigger Event}
}
$$

The model learns:

$$
\boxed{
\text{Audio Sequence}
\rightarrow
\text{Temporal Trigger Probabilities}
}
$$

and is trained with a sequence of binary targets:

$$
\boxed{
\text{Binary Cross-Entropy}
+
\text{BPTT}
}
$$

The major practical challenge is:

$$
\boxed{
\text{Extreme Class Imbalance}
}
$$

while deployment must carefully balance:

$$
\boxed{
\text{False Positives}
\leftrightarrow
\text{False Negatives}
}
$$

The key engineering requirement is:

$$
\boxed{
\text{Small}
+
\text{Fast}
+
\text{Low-Power}
+
\text{Robust}
}
$$

because the detector typically runs continuously and activates a more expensive Speech Recognition system only when the trigger word is detected.